In [1]:
import numpy as np
import json 

    
def load_feature_data(embedding_path, pairwise_path, indices_path):
    embeddings = np.load(embedding_path)
    pairwise_matrix = np.load(pairwise_path)
    with open(indices_path, 'r') as f:
        indices = json.load(f)
    qid_to_idx = {idx: i for i, idx in enumerate(indices)}
    idx_to_qid = {i: idx for i, idx in enumerate(indices)}
    return embeddings, pairwise_matrix,  qid_to_idx, idx_to_qid

def extract_embeddings_for_qids(embeddings, qid_to_idx, qids):
    for qid in qids:
        if qid not in qid_to_idx:
            raise ValueError(f"QID {qid} not found in qid_to_idx mapping.")
    idxs = [qid_to_idx[qid] for qid in qids]
    return embeddings[idxs]

def extract_pairwise_for_qids(pairwise_matrix, qid_to_idx, qids):
    for qid in qids:
        if qid not in qid_to_idx:
            raise ValueError(f"QID {qid} not found in qid_to_idx mapping.")
    idxs = [qid_to_idx[qid] for qid in qids]
    return pairwise_matrix[np.ix_(idxs, idxs)]

embedding_path = "/home/hieunt/verl/data/embedding_data/embeddings_Qwen-Qwen3-Embedding-0.6B_fixprompt-dapo-math-17k_17398.npy"
pairwise_path = "/home/hieunt/verl/data/embedding_data/pairwise_Qwen-Qwen3-Embedding-0.6B_fixprompt-dapo-math-17k_17398_matrix.npy"
indices_path = "/home/hieunt/verl/data/embedding_data/indices_Qwen-Qwen3-Embedding-0.6B_fixprompt-dapo-math-17k_17398.json"
embeddings, pairwise_matrix,  qid_to_idx, idx_to_qid = load_feature_data(embedding_path, pairwise_path, indices_path)

file_path = "/home/hieunt/verl/data/regression_data/allo_grpo_4e/per_question_statistics_latest.json"
with open(file_path, 'r') as f:
    regression_data = json.load(f)
    
    
def get_total_gradients_step(regression_data, batch_size=256):
    total_observations = 0
    for key, value in regression_data.items():
        total_observations += len(value['mean_acc_per_epoch'])
    return total_observations // batch_size

def step_to_epoch_step(step, batch_size=256, total_data_points=17398):
    epoch = (step * batch_size) // total_data_points
    epoch_step = (step * batch_size) % total_data_points
    return epoch, epoch_step

def get_data_at_step(regression_data, step, batch_size=256, target_key='mean_acc_per_epoch'):
    epoch, epoch_step = step_to_epoch_step(step, batch_size)
    all_keys = sorted(regression_data.keys())
    start_idx = epoch_step
    end_idx = min(epoch_step + batch_size, len(all_keys))
    selected_keys = all_keys[start_idx:end_idx]
    data_at_step = {key: regression_data[key][target_key] for key in selected_keys}
    return data_at_step

def prepare_feature_data(data, embedding, pairwise_matrix, qid_to_idx):
    qid_list = [qid for qid in data]
    X = extract_embeddings_for_qids(embedding, qid_to_idx, qid_list)
    P = extract_pairwise_for_qids(pairwise_matrix, qid_to_idx, qid_list)
    y = np.array([data[qid] for qid in qid_list])
    return X, P, y
    
def prepare_time_window_data(regression_data, step, window_size=2, batch_size=256):
    test_data = get_data_at_step(regression_data, step, batch_size)
    train_data = {}
    for train_step in range(max(0, step - window_size), step):
        train_step_data = get_data_at_step(regression_data, train_step, batch_size)
        train_data.update(train_step_data)
    return train_data, test_data

In [2]:
# Compare keys between regression_data and data
reg_keys = regression_data.keys()
data_keys = qid_to_idx.keys()

missing_in_data = list(reg_keys - data_keys)
extra_in_data = list(data_keys - reg_keys)

print(f"Total regression_data keys: {len(reg_keys)}")
print(f"Total data keys:            {len(data_keys)}")
print(f"Missing in data:            {len(missing_in_data)}")
print(f"Extra in data:               {len(extra_in_data)}")

# Preview a few examples
print("\nSample missing (up to 10):", missing_in_data[:10])
print("Sample extra (up to 10):  ", extra_in_data[:10])

Total regression_data keys: 17398
Total data keys:            17398
Missing in data:            0
Extra in data:               0

Sample missing (up to 10): []
Sample extra (up to 10):   []


In [3]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Tuple, Any, Optional

import json
import numpy as np


def _to_str(x: Any) -> str:
    return str(x)


@dataclass
class TimeDataSimulatorConfig:
    embedding_path: str
    pairwise_path: str
    indices_path: str
    regression_json_path: str
    total_data_points: Optional[int] = None  # default: len(indices)


class TimeDataSimulator:
    def __init__(self, config: TimeDataSimulatorConfig):
        self.config = config

        # Load feature artifacts
        self.embeddings = np.load(config.embedding_path)
        self.pairwise_matrix = np.load(config.pairwise_path)
        with open(config.indices_path, "r") as f:
            self.indices: List[Any] = json.load(f)

        # Build mappings; normalize keys to str for robust matching
        self.idx_to_qid: Dict[int, Any] = {i: qid for i, qid in enumerate(self.indices)}
        self.qid_to_idx: Dict[str, int] = {str(qid): i for i, qid in enumerate(self.indices)}

        # Load regression data and normalize keys to str
        with open(config.regression_json_path, "r") as f:
            raw_reg = json.load(f)
        # regression_data: Dict[str, Dict[str, List or Any]]
        self.regression_data: Dict[str, Any] = {str(k): v for k, v in raw_reg.items()}

        # Optionally set total data points from indices if not provided
        self.total_data_points = (
            config.total_data_points if config.total_data_points is not None else len(self.indices)
        )

        # Basic shape checks
        n = len(self.indices)
        if self.embeddings.shape[0] != n:
            raise ValueError(
                f"Embeddings rows {self.embeddings.shape[0]} != indices length {n}"
            )
        if self.pairwise_matrix.shape[0] != n or self.pairwise_matrix.shape[1] != n:
            raise ValueError(
                f"Pairwise shape {self.pairwise_matrix.shape} incompatible with indices length {n}"
            )

    # ----- Step utilities -----
    def get_total_gradient_steps(self, batch_size: int = 256, target_key: str = "mean_acc_per_epoch") -> int:
        total_observations = 0
        for _, value in self.regression_data.items():
            # value[target_key] is expected to be a list/sequence over time
            series = value.get(target_key, [])
            total_observations += len(series)
        return total_observations // batch_size

    def step_to_epoch_step(self, step: int, batch_size: int = 256) -> Tuple[int, int]:
        epoch = (step * batch_size) // self.total_data_points
        epoch_step = (step * batch_size) % self.total_data_points
        return epoch, epoch_step

    def get_data_at_step(
        self,
        step: int,
        batch_size: int = 256,
        target_key: str = "mean_acc_per_epoch",
    ) -> Dict[str, Any]:
        _, epoch_step = self.step_to_epoch_step(step, batch_size)
        # only consider keys present in indices/qid mapping
        keys_filtered = sorted([k for k in self.regression_data.keys() if k in self.qid_to_idx])
        if not keys_filtered:
            return {}

        n = len(keys_filtered)
        bs = min(batch_size, n)
        # cycle over keys to always return a batch
        selected_keys = [keys_filtered[(epoch_step + i) % n] for i in range(bs)]

        # Return the full series for each selected key (mirrors notebook behavior)
        data_at_step = {k: self.regression_data[k][target_key] for k in selected_keys}
        return data_at_step

    def prepare_time_window_data(
        self,
        step: int,
        window_size: int = 2,
        batch_size: int = 256,
        target_key: str = "mean_acc_per_epoch",
    ) -> Tuple[Dict[str, Any], Dict[str, Any]]:
        """Aggregate train over [step - window_size, step) and test at current step.

        Returns (train_dict, test_dict) where each is {qid: value}.
        """
        test_data = self.get_data_at_step(step, batch_size, target_key)
        train_data: Dict[str, Any] = {}
        start = max(0, step - window_size)
        for s in range(start, step):
            d = self.get_data_at_step(s, batch_size, target_key)
            train_data.update(d)
        return train_data, test_data

    # ----- Feature builders -----
    def _extract_embeddings_for_qids(self, qids: List[str]) -> np.ndarray:
        idxs = [self.qid_to_idx[q] for q in qids]
        return self.embeddings[idxs]

    def _extract_pairwise_for_qids(self, qids: List[str]) -> np.ndarray:
        idxs = [self.qid_to_idx[q] for q in qids]
        return self.pairwise_matrix[np.ix_(idxs, idxs)]

    def _filter_known_qids(self, qids: List[Any]) -> Tuple[List[str], List[Any]]:
        """Return (kept_qids_str, kept_qids_original) after filtering to those present in mapping."""
        kept_str: List[str] = []
        kept_orig: List[Any] = []
        for q in qids:
            q_str = _to_str(q)
            if q_str in self.qid_to_idx:
                kept_str.append(q_str)
                kept_orig.append(q)
        return kept_str, kept_orig

    def build_features(self, data: Dict[Any, Any]) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[Any]]:
        """Convert {qid: value} into X, P, y and return also the ordered qids used.

        - If value is a list/sequence, we keep it as-is in y (np.array of objects) or
          consider using a summary (e.g., last, mean). Here we will take the last
          value if it's a non-empty sequence; else np.nan.
        """
        qids_orig: List[Any] = list(data.keys())
        qids_str, qids_kept = self._filter_known_qids(qids_orig)
        if len(qids_str) == 0:
            # Empty set; return consistent empty arrays
            return (
                np.empty((0, self.embeddings.shape[1])),
                np.empty((0, 0)),
                np.empty((0,), dtype=float),
                [],
            )

        X = self._extract_embeddings_for_qids(qids_str)
        P = self._extract_pairwise_for_qids(qids_str)

        y_vals: List[float] = []
        for q in qids_kept:
            v = data[q]
            if isinstance(v, (list, tuple)):
                if len(v) == 0:
                    y_vals.append(np.nan)
                else:
                    y_vals.append(float(v[-1]))  # last value in the series
            else:
                try:
                    y_vals.append(float(v))
                except Exception:
                    y_vals.append(np.nan)
        y = np.asarray(y_vals, dtype=float)
        return X, P, y, qids_kept

    # ----- Public one-shot API -----
    def get_train_test_features(
        self,
        step: int,
        window_size: int = 2,
        batch_size: int = 256,
        target_key: str = "mean_acc_per_epoch",
    ) -> Dict[str, Dict[str, Any]]:
        train_dict, test_dict = self.prepare_time_window_data(
            step=step, window_size=window_size, batch_size=batch_size, target_key=target_key
        )

        X_tr, P_tr, y_tr, qids_tr = self.build_features(train_dict)
        X_te, P_te, y_te, qids_te = self.build_features(test_dict)

        return {
            "train": {"X": X_tr, "P": P_tr, "y": y_tr, "qids": qids_tr},
            "test": {"X": X_te, "P": P_te, "y": y_te, "qids": qids_te},
        }


In [29]:
def distance_delta_y_corr(out: dict, split: str = "train") -> float:
    """Compute correlation between pairwise distances and |Δy| for a given split in `out`.

    Parameters:
    - out: dict with keys 'train'/'test' each containing 'P' (features for distance) and 'y' (labels)
    - split: 'train' or 'test'

    Returns:
    - rho: float correlation coefficient (np.nan if not computable)
    """
    from sklearn.metrics import pairwise_distances
    import numpy as np

    if split not in out or out[split] is None:
        return float("nan")

    P = out[split].get("P")
    y = out[split].get("y")
    if P is None or y is None:
        return float("nan")
    if len(y) < 2:
        return float("nan")

    # pairwise distances and target differences
    D = pairwise_distances(P, metric="euclidean")
    Ydiff = np.abs(y[:, None] - y[None, :])

    # flatten upper triangle for correlation
    idx = np.triu_indices_from(D, k=1)
    x = D[idx]
    z = Ydiff[idx]
    if x.size == 0 or z.size == 0:
        return float("nan")

    rho = np.corrcoef(x, z)[0, 1]
    return float(rho)

def knn_cv_r2(X, y, n_neighbors=10, cv=5, metric="euclidean"):
    """Compute mean cross-validated R² for kNN regressor."""
    from sklearn.neighbors import KNeighborsRegressor
    from sklearn.model_selection import cross_val_score
    import numpy as np
    knn = KNeighborsRegressor(n_neighbors=n_neighbors, metric=metric)
    cv_r2 = cross_val_score(knn, X, y, cv=cv, scoring='r2')
    return float(np.mean(cv_r2))

In [ ]:
# Absolute data paths provided by user
EMBEDDING_PATH = "/home/hieunt/verl/data/embedding_data/embeddings_Qwen-Qwen3-Embedding-0.6B_fixprompt-dapo-math-17k_17398.npy"
PAIRWISE_PATH = "/home/hieunt/verl/data/embedding_data/pairwise_Qwen-Qwen3-Embedding-0.6B_fixprompt-dapo-math-17k_17398_matrix.npy"
INDICES_PATH = "/home/hieunt/verl/data/embedding_data/indices_Qwen-Qwen3-Embedding-0.6B_fixprompt-dapo-math-17k_17398.json"
REGRESSION_PATH = "/home/hieunt/verl/data/regression_data/allo_grpo_4e/per_question_statistics_latest.json"

batch_size = 1000
window_size = 1
target_key = "mean_acc_per_epoch"

# step = 20
for step in range(5, 300, 10):
    cfg = TimeDataSimulatorConfig(
        embedding_path=EMBEDDING_PATH,
        pairwise_path=PAIRWISE_PATH,
        indices_path=INDICES_PATH,
        regression_json_path=REGRESSION_PATH,
        total_data_points=None,
    )

    sim = TimeDataSimulator(cfg)

    # sanity: steps computation doesn't crash
    total_steps = sim.get_total_gradient_steps(batch_size=batch_size, target_key=target_key)
    assert isinstance(total_steps, int)
    assert total_steps >= 0

    out = sim.get_train_test_features(
        step=step, window_size=window_size, batch_size=batch_size, target_key=target_key
    )
    rho = distance_delta_y_corr(out, split="train")
    print(f"step-{step} Distance–|Δy| correlation:", rho)
    print(f"step-{step} kNN CV R²:", knn_cv_r2(out['train']['X'], out['train']['y'], n_neighbors=10, cv=5))


step-5 Distance–|Δy| correlation: 0.02292855366888641
step-5 kNN CV R²: -0.08274552229463653
step-15 Distance–|Δy| correlation: 0.018153442053512343
step-15 kNN CV R²: -0.08159447365925654
step-25 Distance–|Δy| correlation: 0.014026441531509906
step-25 kNN CV R²: -0.09402666764909581
step-35 Distance–|Δy| correlation: 0.012850929145164973
step-35 kNN CV R²: -0.09791008792686402
step-45 Distance–|Δy| correlation: -0.022126821838042775
step-45 kNN CV R²: -0.17235852973577614
step-55 Distance–|Δy| correlation: -0.01302621278674609
step-55 kNN CV R²: -0.10484844661368231
step-65 Distance–|Δy| correlation: 0.012760963719339158
step-65 kNN CV R²: -0.11104615542375462
step-75 Distance–|Δy| correlation: -0.013952189298985595
step-75 kNN CV R²: -0.17243960879432327
step-85 Distance–|Δy| correlation: 0.010478758332416738
step-85 kNN CV R²: -0.22099935634405168
step-95 Distance–|Δy| correlation: 0.02514486006285812
step-95 kNN CV R²: -0.06803878393541973
step-105 Distance–|Δy| correlation: -0.004

In [28]:
# Compute and print kNN CV R² using the helper function
r2 = knn_cv_r2(out['train']['X'], out['train']['y'], n_neighbors=10, cv=5)
print("kNN CV R²:", r2)

kNN CV R²: -0.15422116322944587


In [34]:
def pca_scatter_save(X, y, outdir="pca_plots", fname="pca_scatter.png"):
    """
    Create a PCA scatter plot of X colored by y and save to outdir/fname.
    - X: features (n_samples, n_features)
    - y: targets (n_samples,)
    - outdir: directory to save plot (created if missing)
    - fname: filename for the plot
    """
    import os
    import numpy as np
    from sklearn.decomposition import PCA
    import matplotlib.pyplot as plt

    os.makedirs(outdir, exist_ok=True)
    X_pca = PCA(n_components=2).fit_transform(X)
    plt.figure(figsize=(6, 5))
    sc = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='coolwarm', s=30)
    plt.colorbar(sc, label='y value')
    plt.title('PCA of Features Colored by y')
    plt.xlabel('PCA Component 1')
    plt.ylabel('PCA Component 2')
    plt.tight_layout()
    outpath = os.path.join(outdir, fname)
    plt.savefig(outpath, dpi=120)
    plt.close()
    print(f"Saved PCA scatter plot to {outpath}")

In [64]:
# Absolute data paths provided by user
EMBEDDING_PATH = "/home/hieunt/verl/data/embedding_data/embeddings_Qwen-Qwen3-Embedding-0.6B_fixprompt-dapo-math-17k_17398.npy"
PAIRWISE_PATH = "/home/hieunt/verl/data/embedding_data/pairwise_Qwen-Qwen3-Embedding-0.6B_fixprompt-dapo-math-17k_17398_matrix.npy"
INDICES_PATH = "/home/hieunt/verl/data/embedding_data/indices_Qwen-Qwen3-Embedding-0.6B_fixprompt-dapo-math-17k_17398.json"
REGRESSION_PATH = "/home/hieunt/verl/data/regression_data/allo_grpo_4e/per_question_statistics_latest.json"

batch_size = 256*6
window_size = 1
target_key = "mean_acc_per_epoch"

# step = 20
for step in range(50, 55, 5):
    cfg = TimeDataSimulatorConfig(
        embedding_path=EMBEDDING_PATH,
        pairwise_path=PAIRWISE_PATH,
        indices_path=INDICES_PATH,
        regression_json_path=REGRESSION_PATH,
        total_data_points=None,
    )

    sim = TimeDataSimulator(cfg)

    # sanity: steps computation doesn't crash
    total_steps = sim.get_total_gradient_steps(batch_size=batch_size, target_key=target_key)
    assert isinstance(total_steps, int)
    assert total_steps >= 0

    out = sim.get_train_test_features(
        step=step, window_size=window_size, batch_size=batch_size, target_key=target_key
    )
    rho = distance_delta_y_corr(out, split="train")
    print(f"step-{step} Distance–|Δy| correlation:", rho)
    print(f"step-{step} kNN CV R²:", knn_cv_r2(out['train']['X'], out['train']['y'], n_neighbors=5, cv=5))
    pca_scatter_save(out['train']['X'], out['train']['y'], outdir="pca_plots", fname=f"pca_scatter_step{step}.png")


step-50 Distance–|Δy| correlation: 0.0061914849831449905
step-50 kNN CV R²: -0.19500288049808218
Saved PCA scatter plot to pca_plots/pca_scatter_step50.png


In [59]:
import numpy as np

P = out['train']['P']      # 1280x1280 pairwise distances
y = out['train']['y']      # 1280 values

# Work only with upper triangle to avoid duplicates and self-pairs
idx = np.triu_indices_from(P, k=1)
distances = P[idx]

# Sort by distance descending and take top 10 pairs
top_k = 50
# order = np.argsort(distances)
order = np.argsort(distances)[::-1]
i_idx = idx[0][order[:top_k]]
j_idx = idx[1][order[:top_k]]

for rank, (i, j) in enumerate(zip(i_idx, j_idx), 1):
    print(f"{rank:2d}: pair ({i}, {j})  distance={P[i,j]:.3f}   y[i]={y[i]:.3f}, y[j]={y[j]:.3f}")

dy = np.abs(y[i_idx] - y[j_idx])
threshold = 0.2
# Count how many pairs satisfy |Δy| < threshold
count_similar = np.sum(dy < threshold)

print(f"Among the {top_k} farthest pairs, {count_similar} have |Δy| < {threshold}")
print(f"Fraction: {count_similar / top_k:.3f}")

 1: pair (197, 433)  distance=0.994   y[i]=0.562, y[j]=0.688
 2: pair (178, 184)  distance=0.987   y[i]=0.625, y[j]=0.562
 3: pair (341, 467)  distance=0.871   y[i]=0.375, y[j]=0.562
 4: pair (308, 399)  distance=0.859   y[i]=0.625, y[j]=0.438
 5: pair (174, 302)  distance=0.848   y[i]=0.562, y[j]=0.500
 6: pair (474, 476)  distance=0.840   y[i]=0.500, y[j]=0.562
 7: pair (71, 428)  distance=0.831   y[i]=0.562, y[j]=0.312
 8: pair (337, 356)  distance=0.827   y[i]=0.438, y[j]=0.500
 9: pair (142, 335)  distance=0.822   y[i]=0.312, y[j]=0.562
10: pair (61, 413)  distance=0.821   y[i]=0.438, y[j]=0.375
11: pair (120, 332)  distance=0.820   y[i]=0.625, y[j]=0.562
12: pair (79, 312)  distance=0.816   y[i]=0.688, y[j]=0.625
13: pair (137, 325)  distance=0.815   y[i]=0.750, y[j]=0.375
14: pair (50, 297)  distance=0.814   y[i]=0.375, y[j]=0.375
15: pair (121, 407)  distance=0.810   y[i]=0.625, y[j]=0.688
16: pair (245, 341)  distance=0.810   y[i]=0.250, y[j]=0.375
17: pair (174, 340)  distanc

In [68]:
import numpy as np
from sklearn.metrics import pairwise_distances

def count_topk_pairs_from_X(
    X: np.ndarray,
    y: np.ndarray,
    top_k: int = 50,
    threshold: float = 0.1,
    mode: str = "farthest",   # "nearest" or "farthest"
):
    """
    Compute Euclidean distances from X, pick top-k nearest/farthest pairs (no self / duplicates),
    and count how many have |y[i] - y[j]| < threshold.

    Returns:
        count_similar: int
        frac_similar: float
        pairs_info: list of (i, j, dist, yi, yj, abs_diff) for the selected pairs (sorted)
    """
    assert mode in {"nearest", "farthest"}
    n = X.shape[0]
    # Pairwise Euclidean distances (n x n)
    P = pairwise_distances(X, metric="euclidean", n_jobs=-1)

    # Use only upper triangle (no diagonal, no duplicates)
    iu, ju = np.triu_indices(n, k=1)
    dists = P[iu, ju]

    # Select top_k by distance
    if top_k > len(dists):
        top_k = len(dists)

    if mode == "nearest":
        sel = np.argsort(dists)[:top_k]          # smallest distances
    else:
        sel = np.argsort(dists)[-top_k:][::-1]   # largest distances

    i_idx = iu[sel]
    j_idx = ju[sel]
    d_sel = dists[sel]

    # Compute |Δy|
    dy = np.abs(y[i_idx] - y[j_idx])

    # Count and prepare a small report
    count_similar = int(np.sum(dy < threshold))
    frac_similar = count_similar / top_k if top_k > 0 else 0.0

    # Optional: details for inspection (already sorted by distance)
    pairs_info = [(int(i), int(j), float(d), float(y[i]), float(y[j]), float(abs(y[i]-y[j])))
                  for i, j, d in zip(i_idx, j_idx, d_sel)]

    return count_similar, frac_similar, pairs_info

# ---- Example usage ----
X = out['train']['X']   # shape (num_samples, dim)
y = out['train']['y']   # shape (num_samples,)

top_k = 200
threshold = 0.15

# Farthest pairs
cnt_far, frac_far, far_pairs = count_topk_pairs_from_X(X, y, top_k, threshold, mode="farthest")
print(f"[Farthest] Among top {top_k} pairs, {cnt_far} have |Δy| < {threshold} (fraction={frac_far:.3f})")

# Nearest pairs (often more interesting for locality)
cnt_near, frac_near, near_pairs = count_topk_pairs_from_X(X, y, top_k, threshold, mode="nearest")
print(f"[Nearest ] Among top {top_k} pairs, {cnt_near} have |Δy| < {threshold} (fraction={frac_near:.3f})")

# Optional: peek a few nearest examples
for i, (a, b, d, ya, yb, ddy) in enumerate(near_pairs[:10], 1):
    print(f"Nearest #{i:02d}: ({a},{b}) dist={d:.4f} |Δy|={ddy:.4f}  y[a]={ya:.4f} y[b]={yb:.4f}")

[Farthest] Among top 200 pairs, 135 have |Δy| < 0.15 (fraction=0.675)
[Nearest ] Among top 200 pairs, 131 have |Δy| < 0.15 (fraction=0.655)
Nearest #01: (78,581) dist=0.0000 |Δy|=0.1250  y[a]=0.5625 y[b]=0.4375
Nearest #02: (134,1168) dist=0.0000 |Δy|=0.3125  y[a]=0.7500 y[b]=0.4375
Nearest #03: (883,1529) dist=0.0229 |Δy|=0.0625  y[a]=0.4375 y[b]=0.5000
Nearest #04: (61,620) dist=0.0249 |Δy|=0.1875  y[a]=0.6250 y[b]=0.4375
Nearest #05: (325,328) dist=0.0279 |Δy|=0.0000  y[a]=0.4375 y[b]=0.4375
Nearest #06: (345,1513) dist=0.0298 |Δy|=0.1250  y[a]=0.6250 y[b]=0.7500
Nearest #07: (325,986) dist=0.0451 |Δy|=0.1250  y[a]=0.4375 y[b]=0.5625
Nearest #08: (328,986) dist=0.0489 |Δy|=0.1250  y[a]=0.4375 y[b]=0.5625
Nearest #09: (564,869) dist=0.0586 |Δy|=0.1875  y[a]=0.6250 y[b]=0.4375
Nearest #10: (527,742) dist=0.0634 |Δy|=0.2500  y[a]=0.3750 y[b]=0.6250


In [60]:
out['train']['X'].shape

(512, 1024)

In [67]:
X.shape

(1536, 1024)